In [17]:
import pandas as pd
import random
import csv
from tqdm import tqdm

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

from airouter import AiRouter

client = AiRouter(
   api_key="sk-eXC2ZNKrhHs-9T2Ei1LjOA",
)

In [2]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [3]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(token.lemma_)
        elif token.pos_ == 'VERB':
            verbs.add(token.lemma_)
        elif token.pos_ == 'ADJ':
            adjectives.add(token.lemma_)

In [4]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [5]:
story_features = ['dialoog', 'slecht einde', 'plot twist', 'voorspelling', 'conflict']

In [21]:
models = ["llama-3.3-70b", "gpt-4o-mini", "claude-4-sonnet", "gpt-oss-120b", "mistral-small"]

In [22]:
def generate_prompt():
    chosen_noun = nouns[random.randint(0, len(nouns) - 1)]
    chosen_adjective = adjectives[random.randint(0, len(adjectives) - 1)]
    chosen_verb = verbs[random.randint(0, len(verbs) - 1)]
    idx_features = [random.randint(0, len(story_features) - 1) for _ in range(2)]

    chosen_features = [story_features[i] for i in idx_features]

    prompt = f"""Schrijf een kort verhaal (100-600 woorden) dat vertelt kan worden door een kind tussen de 4 en 12. 
    Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord {chosen_noun} en het volgende bijvoegelijk naamwoord {chosen_adjective}.
    Het verhaal moet de volgende features hebben: {chosen_features[0]} en {chosen_features[1]}.
    """

    return prompt

In [23]:
# Open the CSV once in append mode
# Add model=[...] for hardcoding a model
with open("/Users/sabijn/Documents/PhD/code/storylm_p1_data/results/prompt_ChiSCor_like.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "prompt", "completion"])

    for _ in tqdm(range(650)):
        prompt = generate_prompt()
        response = client.chat.completions.create(
            messages=[
                {"role": "user", "content": f"{prompt}"},
            ],
            weighting={
                "latency": 0.0,
            },
            models=models
        )

        completion = response.choices[0].message.content.strip()

        # Write each row immediately
        writer.writerow([response.model, prompt, completion])


100%|██████████| 650/650 [21:13<00:00,  1.96s/it]
